In [0]:
import requests
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, current_timestamp
from datetime import datetime
from io import StringIO

In [0]:
# Configuration
BASE_URL = "https://www.football-data.co.uk/mmz4281"
PROJECT = "premier_league"
SCHEMA_BRONZE = f"workspace.{PROJECT}_bronze"

# Seasons to download (last 5 seasons) this has many seasons but for the api it has only the current and last year for free usage otherwise you have to pay for it 
SEASONS = {
    "2021": "2020-21",
    "2122": "2021-22",
    "2223": "2022-23",
    "2324": "2023-24", 
    "2425": "2024-25"
}

print("Configuration loaded")
print(f"Target schema: {SCHEMA_BRONZE}")
print(f"Seasons to ingest: {list(SEASONS.values())}")

Configuration loaded
Target schema: workspace.premier_league_bronze
Seasons to ingest: ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25']


In [0]:
def download_season_csv(season_code, season_name):
    """
    Download Premier League match data for a specific season
    
    Args:
        season_code: Season code for URL (e.g., '2324')
        season_name: Human-readable season name (e.g., '2023-24')
    
    Returns:
        DataFrame: PySpark DataFrame with match data
    """
    url = f"{BASE_URL}/{season_code}/E0.csv"
    
    try:
        print(f"\nDownloading {season_name} from: {url}")
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        
        # Read CSV into pandas first
        df_pandas = pd.read_csv(StringIO(response.text))
        
        print(f" Downloaded {len(df_pandas)} matches for {season_name}")
        
        # Convert to Spark DataFrame
        df_spark = spark.createDataFrame(df_pandas)
        
        # Add metadata columns
        df_spark = (df_spark
                    .withColumn("season", lit(season_name))
                    .withColumn("ingestion_date", lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))))
        
        print(f" Converted to Spark DataFrame")
        print(f"  Rows: {df_spark.count()}")
        print(f"  Columns: {len(df_spark.columns)}")
        
        return df_spark
        
    except requests.exceptions.RequestException as e:
        print(f" Error downloading {season_name}: {e}")
        return None
    except Exception as e:
        print(f" Error processing {season_name}: {e}")
        return None

In [0]:
all_seasons = []

for season_code, season_name in SEASONS.items():
    print(f"\n{'='*60}")
    print(f"Processing Season: {season_name}")
    print(f"{'='*60}")
    
    df = download_season_csv(season_code, season_name)
    
    if df is not None:
        # Show sample
        print(f"\nSample data for {season_name}:")
        df.select("Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "season").show(5, truncate=False)
        all_seasons.append(df)

print(f"\n{'='*60}")
print(f" Successfully downloaded {len(all_seasons)} seasons")
print(f"{'='*60}")


Processing Season: 2020-21

 Downloaded 380 matches for 2020-21
 Converted to Spark DataFrame
  Rows: 380
  Columns: 108

Sample data for 2020-21:
+----------+--------------+-----------+----+----+---+-------+
|Date      |HomeTeam      |AwayTeam   |FTHG|FTAG|FTR|season |
+----------+--------------+-----------+----+----+---+-------+
|12/09/2020|Fulham        |Arsenal    |0   |3   |A  |2020-21|
|12/09/2020|Crystal Palace|Southampton|1   |0   |H  |2020-21|
|12/09/2020|Liverpool     |Leeds      |4   |3   |H  |2020-21|
|12/09/2020|West Ham      |Newcastle  |0   |2   |A  |2020-21|
|13/09/2020|West Brom     |Leicester  |0   |3   |A  |2020-21|
+----------+--------------+-----------+----+----+---+-------+
only showing top 5 rows

Processing Season: 2021-22

 Downloaded 380 matches for 2021-22
 Converted to Spark DataFrame
  Rows: 380
  Columns: 108

Sample data for 2021-22:
+----------+----------+--------------+----+----+---+-------+
|Date      |HomeTeam  |AwayTeam      |FTHG|FTAG|FTR|season |


In [0]:
if all_seasons:
    # Union all seasons using unionByName to handle column mismatches
    combined_df = all_seasons[0]
    for df in all_seasons[1:]:
        combined_df = combined_df.unionByName(df, allowMissingColumns=True)
    
    print(f"\nCombined Dataset Summary:")
    print(f"Total matches: {combined_df.count()}")
    print(f"Total columns: {len(combined_df.columns)}")
    print(f"Seasons included: {', '.join(SEASONS.values())}")
    
    # Show column count breakdown
    print(f"\nNote: Different seasons have different column counts:")
    print(f"  2020-21 to 2023-24: ~108 columns")
    print(f"  2024-25: ~122 columns")
    print(f"  Combined table: {len(combined_df.columns)} columns (union of all)")
    
    # Save to Delta table
    target_table = f"{SCHEMA_BRONZE}.pl_matches_historical"
    
    print(f"\nSaving to Delta table: {target_table}")
    
    (combined_df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(target_table))
    
    print(f" Saved to Databricks workspace table: {target_table}")
    print(f"  View in Catalog → workspace → premier_league_bronze → pl_matches_historical")
    
else:
    print("✗ No data to save")


Combined Dataset Summary:
Total matches: 1900
Total columns: 135
Seasons included: 2020-21, 2021-22, 2022-23, 2023-24, 2024-25

Note: Different seasons have different column counts:
  2020-21 to 2023-24: ~108 columns
  2024-25: ~122 columns
  Combined table: 135 columns (union of all)

Saving to Delta table: workspace.premier_league_bronze.pl_matches_historical
✓ Saved to Databricks workspace table: workspace.premier_league_bronze.pl_matches_historical
  View in Catalog → workspace → premier_league_bronze → pl_matches_historical


In [0]:
df_verify = spark.read.table(f"{SCHEMA_BRONZE}.pl_matches_historical")

print("\nMatches per Season:")
df_verify.groupBy("season").count().orderBy("season").show()


Matches per Season:
+-------+-----+
| season|count|
+-------+-----+
|2020-21|  380|
|2021-22|  380|
|2022-23|  380|
|2023-24|  380|
|2024-25|  380|
+-------+-----+



In [0]:
print("\nSample of saved data (first 10 matches):")
df_verify.select("Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR", "season").show(10, truncate=False)

print(f"\nTotal columns in combined table: {len(df_verify.columns)}")
print("\nColumn names (first 30):")
for i, col in enumerate(df_verify.columns[:30], 1):
    print(f"  {i}. {col}")
if len(df_verify.columns) > 30:
    print(f"  ... and {len(df_verify.columns) - 30} more columns")


Sample of saved data (first 10 matches):
+----------+----------------+-----------+----+----+---+-------+
|Date      |HomeTeam        |AwayTeam   |FTHG|FTAG|FTR|season |
+----------+----------------+-----------+----+----+---+-------+
|12/09/2020|Fulham          |Arsenal    |0   |3   |A  |2020-21|
|12/09/2020|Crystal Palace  |Southampton|1   |0   |H  |2020-21|
|12/09/2020|Liverpool       |Leeds      |4   |3   |H  |2020-21|
|12/09/2020|West Ham        |Newcastle  |0   |2   |A  |2020-21|
|13/09/2020|West Brom       |Leicester  |0   |3   |A  |2020-21|
|13/09/2020|Tottenham       |Everton    |0   |1   |A  |2020-21|
|14/09/2020|Brighton        |Chelsea    |1   |3   |A  |2020-21|
|14/09/2020|Sheffield United|Wolves     |0   |2   |A  |2020-21|
|19/09/2020|Everton         |West Brom  |5   |2   |H  |2020-21|
|19/09/2020|Leeds           |Fulham     |4   |3   |H  |2020-21|
+----------+----------------+-----------+----+----+---+-------+
only showing top 10 rows

Total columns in combined table: 135

In [0]:
# Check for key columns (present in all seasons)
key_columns = ["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]

print("\nData Quality Check:")
print(f"Total rows: {df_verify.count()}")

print("\nMissing values in key columns:")
for col in key_columns:
    null_count = df_verify.filter(df_verify[col].isNull()).count()
    print(f"  {col}: {null_count} missing")


Data Quality Check:
Total rows: 1900

Missing values in key columns:
  Date: 0 missing
  HomeTeam: 0 missing
  AwayTeam: 0 missing
  FTHG: 0 missing
  FTAG: 0 missing
  FTR: 0 missing
